# 08 — Stacking (nested, out-of-fold)

**Purpose.** Test whether combining several models beats the best single model, using a procedure that cannot leak: the meta-learner is trained only on predictions the base models made for patients they had not seen.

**Reads:** `../data/raw/dataset.xlsx`, `../results/tune_best_per_fold.csv`, `../results/tune_summary.csv` · **Writes:** `../results/stack_*.csv`, `stack_oof.npz`

**Fixed by earlier notebooks:** 14 raw features (06), no class weighting (05), same 5 outer folds (04–07), Random Forest at defaults and XGBoost tuned (07).

**How stacking works here (per outer fold)**
1. **Inner out-of-fold predictions.** Inside the outer training part, 5 inner folds: each base model predicts class probabilities for patients it was *not* trained on. This gives every outer-training patient a clean 5 × 7 = 35-number "meta-feature" row (5 base models × 7 classes).
2. **Meta-learner.** A multinomial logistic regression learns from those rows how much to trust each base model for each class. Its regularisation strength `C` is chosen by 5-fold CV on the meta-features (macro-F1).
3. **Outer prediction.** Each base model is refit on the whole outer training part, predicts the outer validation patients, and the meta-learner combines them. The outer validation patients are never seen by any base model or by the meta-learner during training.

**Base models** (all unweighted, 14 raw features): Logistic Regression (linear; the most different from trees), k-nearest neighbours (k = 25, distance-weighted; local), Random Forest (defaults), **XGBoost with the settings tuned for this fold in notebook 07** (selected on that fold's training part only, so no leakage), HistGradientBoosting (a fast, untuned, independent boosting implementation).

**Why XGBoost uses per-fold settings and not the final settings.** The final XGBoost settings of notebook 07 were found on the *entire* training pool, including the validation patients; using them here would leak hyper-parameter selection into the estimate. The per-fold settings were chosen without those patients, so this notebook reproduces notebook 07's nested XGBoost result exactly (checked in section 4).

**Configurations compared:** each of the 5 base models alone; `avg_all` (plain average of all 5 probability vectors); `avg_trees` (average of the three tree ensembles); `stack_lr` (the learned meta-learner).

**Pre-specified decision rule.**
* The reference is **XGBoost (tuned)**, fixed in advance: it has the best nested macro-F1 and the best calibration among single models.
* `stack_lr` replaces it only if the 95% paired-bootstrap interval of the macro-F1 gain **excludes 0** *and* it wins in **at least 4 of 5 folds**. Otherwise the single tuned XGBoost remains the final model.
* The other comparisons (averages vs stack, RF vs XGBoost) are reported for the paper but do not decide anything.
* The meta-learner is fixed in advance as logistic regression (only `C` is tuned, inside the training part); no alternative meta-learner is chosen by looking at outer scores.

**Expectation, stated in advance.** Three of the five base models are tree ensembles with highly correlated errors, and all single models already sit within about 0.003 of each other. Any stacking gain is expected to be below 0.003, i.e. not adoptable. The cost of adopting a stack is real (five models to refit, one more thing to explain), so a null result is the likely and useful outcome.

**Caveat.** As in notebook 07, the imputer/scaler of each outer fold saw the inner validation rows; this is a mild inner-loop optimism that does not affect the outer scores.

## 0. Setup

**What this does:** imports libraries and the shared helpers, prints versions.

In [1]:
import sys, os, time, json, warnings
sys.path.append('..')

import numpy as np
import pandas as pd
import sklearn, xgboost
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

from src.preprocessing.cv_pipeline import preprocess_fold, load_cv_pool
from src.evaluation.metrics import (macro_f1_fast, per_class_prf, brier_multiclass, ece_top_label,
                                    mcnemar_exact, holm, bootstrap_f1_matrix)

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)
os.makedirs('../results', exist_ok=True)

pandas 3.0.5 | numpy 2.5.2 | scikit-learn 1.9.0 | xgboost 3.4.1


## 1. Load the CV pool, the folds and the per-fold XGBoost settings

**What this does:** loads the 9,668 training patients (never the test set), verifies them against notebook 02, recreates the 5 outer folds, and reads from notebook 07 the XGBoost settings that won in each outer fold's inner search.

**Checks:** same pool and folds as before; one XGBoost setting per fold is available.

In [2]:
pool = load_cv_pool("../data/raw/dataset.xlsx")
ref_idx = pd.read_csv('../data/processed/y_train.csv', index_col=0).index
assert pool.shape == (9668, 17) and pool.index.equals(ref_idx)
y_global = LabelEncoder().fit_transform(pool['Disease'])
classes = sorted(pool['Disease'].unique()); K = len(classes)
AS, REA = classes.index('Ankylosing Spondylitis'), classes.index('Reactive Arthritis')

N_SPLITS = 5
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42).split(pool, pool['Disease']))
cover = np.zeros(len(pool), int)
for tr, va in folds:
    cover[va] += 1
assert (cover == 1).all()

best07 = pd.read_csv('../results/tune_best_per_fold.csv')
xgb_params = {int(r['fold']): json.loads(r['best_params']) for _, r in best07[best07.model == 'XGBoost'].iterrows()}
assert sorted(xgb_params) == [1, 2, 3, 4, 5]
print("pool OK | fold sizes:", [(len(tr), len(va)) for tr, va in folds])
print("XGBoost settings per fold loaded from notebook 07:", {f: p['learning_rate'] for f, p in xgb_params.items()}, "(learning rates)")

pool OK | fold sizes: [(7734, 1934), (7734, 1934), (7734, 1934), (7735, 1933), (7735, 1933)]
XGBoost settings per fold loaded from notebook 07: {1: 0.026428173189806673, 2: 0.021797167556513685, 3: 0.0269814361896173, 4: 0.022903641944526563, 5: 0.08466653703650753} (learning rates)


## 2. Base models and configurations

**What this does:** defines the five base models. Everything except XGBoost uses fixed settings (no tuning). `HistGradientBoosting` runs with `early_stopping=False` so its behaviour does not depend on an internal random split.

**Configurations scored on the outer validation patients:** the 5 base models, `avg_all`, `avg_trees` (Random Forest + XGBoost + HistGradientBoosting) and `stack_lr`.

In [3]:
base = {
    'Logistic Regression':  lambda f: LogisticRegression(max_iter=2000, random_state=42),
    'KNN':                  lambda f: KNeighborsClassifier(n_neighbors=25, weights='distance', n_jobs=-1),
    'Random Forest':        lambda f: RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'XGBoost (tuned)':      lambda f: XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', random_state=42, n_jobs=-1, **xgb_params[f]),
    'HistGradientBoosting': lambda f: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early_stopping=False, random_state=42),
}
names = list(base)
TREES = ['Random Forest', 'XGBoost (tuned)', 'HistGradientBoosting']
configs = names + ['avg_all', 'avg_trees', 'stack_lr']
print("base models:", names)
print("configurations:", configs)

base models: ['Logistic Regression', 'KNN', 'Random Forest', 'XGBoost (tuned)', 'HistGradientBoosting']
configurations: ['Logistic Regression', 'KNN', 'Random Forest', 'XGBoost (tuned)', 'HistGradientBoosting', 'avg_all', 'avg_trees', 'stack_lr']


## 3. The nested stacking loop

**What this does, for each outer fold:** builds the 35-column meta-features (section header steps 1–3), fits the meta-learner, and scores every configuration on the outer validation patients. Also stores, for the diagnostics, the meta-learner's chosen `C` and how much weight it gives each base model.

**Runtime:** roughly 10–15 minutes (30 model fits per fold: 5 base models × 5 inner folds + 5 refits). One progress line per fold.

**Checks per fold:** no missing values after preprocessing; encoded labels equal the global encoding; every base model has classes 0–6; meta-feature rows are valid probabilities (each 7-block sums to 1).

In [4]:
oof = {c: np.zeros((len(pool), K)) for c in configs}
y_all = np.full(len(pool), -1)
fold_rows, meta_rows = [], []

def record(fold, cfg, P, yva):
    pred = P.argmax(1)
    pr, rc, f1 = per_class_prf(yva, pred, K)
    fold_rows.append({'fold': fold, 'config': cfg, 'macro_f1': f1_score(yva, pred, average='macro'),
                      'balanced_accuracy': rc.mean(), 'AS_f1': f1[AS], 'ReA_f1': f1[REA]})

for fold, (tr, va) in enumerate(folds, 1):
    t0 = time.time()
    ft, fv = pool.iloc[tr], pool.iloc[va]
    Xr_tr, Xr_va, _, _, y_tr, y_va = preprocess_fold(ft, fv)
    assert Xr_tr.isna().sum().sum() == 0 and Xr_va.isna().sum().sum() == 0 and Xr_tr.shape[1] == 14
    assert (y_va.to_numpy() == y_global[va]).all() and (y_tr.to_numpy() == y_global[tr]).all()
    Xt, Xv, ytr, yva = Xr_tr, Xr_va, y_tr.to_numpy(), y_va.to_numpy()
    y_all[va] = yva
    inner = list(StratifiedKFold(5, shuffle=True, random_state=1).split(Xt, ytr))
    Ztr = np.zeros((len(ytr), K * len(names))); Zva = np.zeros((len(yva), K * len(names)))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for b, nm in enumerate(names):
            sl = slice(b * K, (b + 1) * K)
            for itr, iva in inner:
                m = base[nm](fold).fit(Xt.iloc[itr], ytr[itr])
                assert list(m.classes_) == list(range(K))
                Ztr[iva, sl] = m.predict_proba(Xt.iloc[iva])
            m = base[nm](fold).fit(Xt, ytr)
            assert list(m.classes_) == list(range(K))
            Zva[:, sl] = m.predict_proba(Xv)
        for Z in (Ztr, Zva):
            assert np.allclose(Z.reshape(len(Z), len(names), K).sum(2), 1.0, atol=1e-6)
        blocks = {nm: Zva[:, b * K:(b + 1) * K] for b, nm in enumerate(names)}
        for nm in names:
            oof[nm][va] = blocks[nm]; record(fold, nm, blocks[nm], yva)
        P_all = np.mean([blocks[nm] for nm in names], axis=0)
        P_trees = np.mean([blocks[nm] for nm in TREES], axis=0)
        meta = LogisticRegressionCV(Cs=[0.01, 0.1, 1, 10, 100], cv=StratifiedKFold(5, shuffle=True, random_state=0),
                                    scoring='f1_macro', max_iter=3000, random_state=0, n_jobs=-1).fit(Ztr, ytr)
        P_stack = meta.predict_proba(Zva)
    for cfg, P in (('avg_all', P_all), ('avg_trees', P_trees), ('stack_lr', P_stack)):
        oof[cfg][va] = P; record(fold, cfg, P, yva)
    w = np.abs(meta.coef_).reshape(K, len(names), K).sum(axis=(0, 2))
    meta_rows.append({'fold': fold, 'chosen_C': float(meta.C_[0]), **{nm: float(w[i] / w.sum()) for i, nm in enumerate(names)}})
    print(f"fold {fold}/{N_SPLITS} done in {time.time() - t0:.0f}s (stack_lr C={meta.C_[0]})")

fold_df = pd.DataFrame(fold_rows); meta_df = pd.DataFrame(meta_rows)
oof_pred = {k: v.argmax(1) for k, v in oof.items()}
print("done.")

fold 1/5 done in 199s (stack_lr C=0.1)
fold 2/5 done in 50s (stack_lr C=0.1)
fold 3/5 done in 58s (stack_lr C=1.0)
fold 4/5 done in 60s (stack_lr C=0.1)
fold 5/5 done in 47s (stack_lr C=0.1)
done.


## 4. Integrity checks and reproduction of notebook 07

**What this does:**
* every patient has a label; every probability row sums to 1;
* **Random Forest (defaults)** and **XGBoost (tuned, per-fold settings)** must reproduce notebook 07's `default` and `tuned` nested results exactly (same folds, features, models, settings).

**If any difference exceeds 0.0005, stop and investigate** — the comparison against the reference would not be trustworthy.

In [5]:
assert (y_all == y_global).all()
for k, P in oof.items():
    assert np.allclose(P.sum(1), 1.0, atol=1e-6), k

t07 = pd.read_csv('../results/tune_summary.csv').set_index(['model', 'variant'])['macro_f1_mean']
gmean = fold_df.groupby('config').macro_f1.mean()
rep = pd.DataFrame([
    {'check': 'Random Forest (defaults)', 'notebook_07': t07[('Random Forest', 'default')], 'this_notebook': gmean['Random Forest']},
    {'check': 'XGBoost (tuned, per fold)', 'notebook_07': t07[('XGBoost', 'tuned')], 'this_notebook': gmean['XGBoost (tuned)']}])
rep['diff'] = rep['this_notebook'] - rep['notebook_07']
print(rep.round(5).to_string(index=False))
print("\nmax |diff| =", rep['diff'].abs().max().round(6))
if rep['diff'].abs().max() > 0.0005:
    print("WARNING: does not reproduce notebook 07 - investigate.")

                    check  notebook_07  this_notebook  diff
 Random Forest (defaults)      0.82303        0.82303   0.0
XGBoost (tuned, per fold)      0.82544        0.82544   0.0

max |diff| = 0.0


## 5. Results table

**What this does:** one row per configuration, sorted by mean fold macro-F1: 5-fold mean ± std, the pooled out-of-fold score with a 95% bootstrap interval (2,000 resamples), the two problem classes, balanced accuracy, and calibration (Brier, ECE; lower is better).

**How to read it:** as before, neighbouring rows are usually within the interval. The comparisons in section 7 decide what counts.

In [6]:
keys_b, F = bootstrap_f1_matrix(y_all, oof_pred, B=2000, seed=42)
kpos = {k: j for j, k in enumerate(keys_b)}
gk = fold_df.groupby('config')
rows_s = []
for k in configs:
    pr, rc, f1 = per_class_prf(y_all, oof_pred[k], K)
    rows_s.append({'config': k, 'macro_f1_mean': gk.macro_f1.mean()[k], 'macro_f1_std': gk.macro_f1.std()[k],
                   'oof_macro_f1': macro_f1_fast(y_all, oof_pred[k]),
                   'ci_low': np.percentile(F[:, kpos[k]], 2.5), 'ci_high': np.percentile(F[:, kpos[k]], 97.5),
                   'AS_f1': f1[AS], 'ReA_f1': f1[REA], 'balanced_acc': rc.mean(),
                   'brier': brier_multiclass(y_all, oof[k]), 'ece': ece_top_label(y_all, oof[k])})
summary = pd.DataFrame(rows_s).sort_values('macro_f1_mean', ascending=False).reset_index(drop=True)
summary.round(4)

,config,macro_f1_mean,macro_f1_std,oof_macro_f1,ci_low,ci_high,AS_f1,ReA_f1,balanced_acc,brier,ece
0,stack_lr,0.8260,0.0051,0.8261,0.8178,0.8343,0.6706,0.6685,0.8159,0.2156,0.0382
1,avg_all,0.8259,0.0056,0.8259,0.8174,0.8346,0.6709,0.6600,0.8135,0.2194,0.0360
2,XGBoost (tuned),0.8254,0.0078,0.8254,0.8168,0.8336,0.6671,0.6703,0.8159,0.2098,0.0153
3,Random Forest,0.8230,0.0070,0.8230,0.8139,0.8316,0.6623,0.6563,0.8114,0.2196,0.0370
4,avg_trees,0.8227,0.0056,0.8226,0.8142,0.8311,0.6631,0.6547,0.8127,0.2141,0.0330
5,HistGradientBoosting,0.8125,0.0051,0.8126,0.8041,0.8212,0.6553,0.6285,0.8053,0.2579,0.0994
6,Logistic Regression,0.7889,0.0047,0.7891,0.7797,0.7981,0.6354,0.5371,0.7775,0.2629,0.0113
7,KNN,0.7058,0.0136,0.7061,0.6983,0.7142,0.5588,0.0804,0.7132,0.2953,0.0670


## 6. Why does (or doesn't) stacking help? Diagnostics

**6a. Disagreement between base models.** The share of patients on which two base models predict a *different* class. Stacking can only help when models disagree *and* the disagreements are informative; if the tree ensembles disagree on only ~5% of patients, there is little to combine.

**6b. Oracle ceiling.** The accuracy you would get if you could always pick the right base model (the patient counts as correct if *any* base model is right) versus the best single model. A large gap means the models are complementary *in principle*; it is an upper bound, never achievable in practice.

**6c. What the meta-learner relies on.** The share of the meta-learner's total absolute weight given to each base model (mean over folds), plus the regularisation `C` it chose. A meta-learner that puts most weight on one model is just re-learning that model.

In [7]:
D = pd.DataFrame({a: {b: float((oof_pred[a] != oof_pred[b]).mean()) for b in names} for a in names}).round(3)
print("6a. Disagreement rate between base models (out-of-fold):")
print(D.to_string())

correct = np.column_stack([(oof_pred[nm] == y_all) for nm in names])
best_single = max(correct.mean(0))
print(f"\n6b. Oracle accuracy (any base model right): {correct.any(1).mean():.4f} | best single accuracy: {best_single:.4f}")

print("\n6c. Meta-learner weight shares (mean over folds) and chosen C:")
print(meta_df[['fold', 'chosen_C'] + names].round(3).to_string(index=False))
print(meta_df[names].mean().round(3).to_string())

6a. Disagreement rate between base models (out-of-fold):
                      Logistic Regression    KNN  Random Forest  XGBoost (tuned)  HistGradientBoosting
Logistic Regression                 0.000  0.149          0.121            0.121                 0.149
KNN                                 0.149  0.000          0.128            0.138                 0.161
Random Forest                       0.121  0.128          0.000            0.048                 0.078
XGBoost (tuned)                     0.121  0.138          0.048            0.000                 0.077
HistGradientBoosting                0.149  0.161          0.078            0.077                 0.000

6b. Oracle accuracy (any base model right): 0.9161 | best single accuracy: 0.8390

6c. Meta-learner weight shares (mean over folds) and chosen C:
 fold  chosen_C  Logistic Regression   KNN  Random Forest  XGBoost (tuned)  HistGradientBoosting
    1       0.1                0.190 0.169          0.206            0.251       

## 7. Comparisons and the pre-specified decision

**What this does:** for each comparison, `diff` = pooled out-of-fold macro-F1 of A minus B, its 95% paired-bootstrap interval, the number of folds where A beats B, and McNemar's exact test on accuracy (`b` = only A right, `c` = only B right; Holm-corrected across the comparisons).

**The decisive comparison** is `stack_lr` vs `XGBoost (tuned)`. It passes only if `ci_low > 0` **and** `folds_A_better ≥ 4`. The other rows are informational: does learning the weights beat plain averaging, does averaging beat the reference, and which single tree model is better (this tells us whether the final model should be RF or XGBoost if no stack is adopted).

In [8]:
ref = 'XGBoost (tuned)'
piv = fold_df.pivot(index='fold', columns='config', values='macro_f1')
comparisons = [('stack_lr', ref, 'DECISIVE: stack vs reference'),
               ('avg_trees', ref, 'average of trees vs reference'),
               ('avg_all', ref, 'average of all 5 vs reference'),
               ('stack_lr', 'avg_trees', 'stack vs plain average of trees'),
               ('stack_lr', 'avg_all', 'stack vs plain average of all 5'),
               (ref, 'Random Forest', 'XGBoost (tuned) vs Random Forest (defaults)')]
out = []
for A, B, label in comparisons:
    d = F[:, kpos[A]] - F[:, kpos[B]]
    b, c, p = mcnemar_exact(y_all, oof_pred[A], oof_pred[B])
    fd = piv[A] - piv[B]
    out.append({'comparison': label, 'A': A, 'B': B, 'diff': macro_f1_fast(y_all, oof_pred[A]) - macro_f1_fast(y_all, oof_pred[B]),
                'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5), 'folds_A_better': int((fd > 0).sum()),
                'mcnemar_b': b, 'mcnemar_c': c, 'p_raw': p})
cmp_df = pd.DataFrame(out)
cmp_df['p_holm'] = holm(cmp_df['p_raw'])
print(cmp_df.drop(columns=['A', 'B', 'p_raw']).round(4).to_string(index=False))

dec = cmp_df.iloc[0]
stack_wins = bool(dec['ci_low'] > 0 and dec['folds_A_better'] >= 4)
print(f"\nStack beats the reference (interval excludes 0 and >= 4/5 folds): {stack_wins}")
print("FINAL MODEL DECISION:", "stack_lr (5 base models + logistic-regression meta-learner)" if stack_wins else "XGBoost (tuned) alone")

                                 comparison    diff  ci_low  ci_high  folds_A_better  mcnemar_b  mcnemar_c  p_holm
               DECISIVE: stack vs reference  0.0007 -0.0032   0.0045               3        170        178     1.0
              average of trees vs reference -0.0028 -0.0073   0.0015               1        198        225     1.0
              average of all 5 vs reference  0.0005 -0.0041   0.0053               4        198        195     1.0
            stack vs plain average of trees  0.0035 -0.0011   0.0079               4        261        242     1.0
            stack vs plain average of all 5  0.0002 -0.0039   0.0043               3        166        177     1.0
XGBoost (tuned) vs Random Forest (defaults)  0.0024 -0.0025   0.0071               4        225        217     1.0

Stack beats the reference (interval excludes 0 and >= 4/5 folds): False
FINAL MODEL DECISION: XGBoost (tuned) alone


## 8. Save the results

**What this does:** writes all tables and the out-of-fold probabilities to `../results/`.

In [9]:
fold_df.to_csv('../results/stack_fold_scores.csv', index=False)
summary.to_csv('../results/stack_summary.csv', index=False)
cmp_df.drop(columns=['p_raw']).to_csv('../results/stack_comparisons.csv', index=False)
meta_df.to_csv('../results/stack_meta_weights.csv', index=False)
np.savez_compressed('../results/stack_oof.npz', y=y_all, classes=np.array(classes),
                    **{k.replace(' ', '_').replace('(', '').replace(')', ''): v.astype(np.float32) for k, v in oof.items()})
print(sorted(f for f in os.listdir('../results') if f.startswith('stack')))

['stack_comparisons.csv', 'stack_fold_scores.csv', 'stack_meta_weights.csv', 'stack_oof.npz', 'stack_summary.csv']


## Summary — what this notebook establishes

* Whether a stacked ensemble beats the single tuned XGBoost by more than noise (section 7), judged on patients no base model or meta-learner saw.
* Why (section 6): how much the models disagree, how much complementary signal exists in principle, and what the meta-learner actually uses.
* The **final model definition for notebook 10**: either XGBoost with the final settings from `tuning_final_params.json`, or the stack, whose base models would be refit on all 9,668 training patients with the meta-learner trained on their 5-fold out-of-fold predictions.